In [1]:
# Q2: Drift-Robust Quantum Machine Learning for Electronic Nose Sensor Classification
# One Kaggle cell. Attach UCI Gas Sensor Array Drift ZIP or enable Internet.
import os, sys, subprocess, pathlib, zipfile, io, re, json, time, random, warnings
warnings.filterwarnings('ignore')
for mod,pkg in [('pennylane','pennylane'),('requests','requests')]:
 try: __import__(mod)
 except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
import numpy as np, pandas as pd, torch, torch.nn as nn, pennylane as qml, requests
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from pathlib import Path
SEED=42; N_QUBITS=4; LAYERS=2; QSVM_TRAIN=240; TEST_CAP=240; VQC_TRAIN=500; VQC_EPOCHS=12; BATCH_SIZE=16
OUT=Path('/kaggle/working/Q2_Drift_Robust_QML'); OUT.mkdir(parents=True,exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('PyTorch:',torch.__version__,'PennyLane:',qml.__version__,'CUDA available:',torch.cuda.is_available(),'(quantum simulator uses CPU)')
def nested_zip(data,depth=0):
 if depth>4:return []
 result=[]
 with zipfile.ZipFile(io.BytesIO(data)) as z:
  for name in z.namelist():
   if name.lower().endswith('.dat'):result.append((name,z.read(name)))
   elif name.lower().endswith('.zip'):result+=nested_zip(z.read(name),depth+1)
 return result
files=[]
for p in Path('/kaggle/input').rglob('*'):
 if not p.is_file():continue
 try:
  if p.suffix.lower()=='.dat':files.append((str(p),p.read_bytes()))
  elif p.suffix.lower()=='.zip':files+=nested_zip(p.read_bytes())
 except Exception:pass
if not files:
 for url in ['https://archive.ics.uci.edu/static/public/224/gas+sensor+array+drift+dataset.zip','https://archive.ics.uci.edu/ml/machine-learning-databases/00224/Dataset.zip']:
  try:
   r=requests.get(url,timeout=90);r.raise_for_status();files=nested_zip(r.content)
   if files:break
  except Exception as e:print('Download attempt:',str(e)[:100])
if not files:raise RuntimeError('Attach official UCI Gas Sensor Array Drift dataset ZIP to Kaggle or enable Internet.')
rows=[]; labels=[]; batches=[]; seen=set()
for name,blob in files:
 match=re.search(r'batch[_\s-]*(\d+)',name,re.I)
 if not match:continue
 b=int(match.group(1))
 # Ignore duplicate copies of the same batch file
 if b in seen:continue
 seen.add(b)
 for line in blob.decode('utf-8',errors='replace').splitlines():
  t=line.strip().split()
  if len(t)<2:continue
  try:
   v=np.zeros(128,dtype=np.float32)
   for term in t[1:]:
    if ':' not in term:continue
    idx,val=term.split(':',1);idx=int(idx)-1
    if 0<=idx<128:v[idx]=float(val)
   rows.append(v);labels.append(int(float(t[0])));batches.append(b)
  except (ValueError,IndexError):pass
X=np.asarray(rows,dtype=np.float32);yraw=np.asarray(labels);batch=np.asarray(batches)
assert len(X)>0 and len(np.unique(yraw))==6,'Check dataset files and labels'
classes=np.sort(np.unique(yraw));y=np.searchsorted(classes,yraw)
print('Loaded:',X.shape,'batches:',sorted(set(batch)),'class counts:',np.bincount(y))
assert all(np.isin([1,2,3,4,5,6,7,8,9,10],np.unique(batch))),'All 10 batch files are required.'
# Fixed chronological evaluation: train 1-7, validation 8, test 9-10
train_idx=np.where(batch<=7)[0];val_idx=np.where(batch==8)[0];test_idx=np.where(batch>=9)[0]
# Subset selection is reproducible; stratify when possible.
def cap(idx,n):
 if len(idx)<=n:return idx
 try:a,_=train_test_split(idx,train_size=n,stratify=y[idx],random_state=SEED);return np.sort(a)
 except ValueError:return np.sort(np.random.default_rng(SEED).choice(idx,n,replace=False))
qtrain=cap(train_idx,QSVM_TRAIN);vtrain=cap(train_idx,VQC_TRAIN);qtest=cap(test_idx,TEST_CAP)
# Preprocessing learned on chronological training ONLY; same 4-D features for all models.
imp=SimpleImputer(strategy='median');sc=StandardScaler();pca=PCA(n_components=N_QUBITS,random_state=SEED)
A=imp.fit_transform(X[train_idx]);A=sc.fit_transform(A);pca.fit(A)
def features(idx):return pca.transform(sc.transform(imp.transform(X[idx]))).astype('float32')
# Bound angles for numerical stability; parameters fitted on train only.
Ftrain=features(train_idx);scale=np.maximum(np.quantile(np.abs(Ftrain),0.95,axis=0),1e-6)
def angles(idx):return (np.pi*np.tanh(features(idx)/scale)).astype('float32')
Xt=angles(qtrain);Xv=angles(val_idx);Xe=angles(qtest)
records=[];predictions=[]
def record(model,truth,pred,secs,idx):
 a=accuracy_score(truth,pred);f=f1_score(truth,pred,average='macro',zero_division=0)
 records.append(dict(model=model,accuracy=a,macro_f1=f,seconds=secs,n_train=QSVM_TRAIN if model!='VQC' else len(vtrain),n_test=len(truth)))
 predictions.extend(dict(model=model,index=int(i),batch=int(batch[i]),actual=int(t),predicted=int(p)) for i,t,p in zip(idx,truth,pred))
 print(f'{model}: accuracy={a:.4f}, macro_F1={f:.4f}, time={secs:.1f}s')
 (OUT/f'{model}_report.txt').write_text(classification_report(truth,pred,labels=np.arange(6),zero_division=0))
 fig,ax=plt.subplots(figsize=(6,5));ConfusionMatrixDisplay(confusion_matrix(truth,pred,labels=np.arange(6))).plot(ax=ax,cmap='Blues',colorbar=False)
 ax.set_title(model+' chronological drift');fig.tight_layout();fig.savefig(OUT/f'{model}_confusion.png',dpi=160);plt.close(fig)
# Classical SVM matched to QSVM subset
start=time.time();svm=SVC(C=1,kernel='rbf',gamma='scale');svm.fit(Xt,y[qtrain]);pred=svm.predict(Xe)
record('SVM',y[qtest],pred,time.time()-start,qtest)
# QSVM quantum fidelity kernel: pairwise state overlaps. State vectors cached once per sample.
qdev=qml.device('default.qubit',wires=N_QUBITS)
@qml.qnode(qdev)
def state(x):
 qml.AngleEmbedding(x,wires=range(N_QUBITS),rotation='Y')
 for i in range(N_QUBITS):qml.CNOT(wires=[i,(i+1)%N_QUBITS])
 qml.AngleEmbedding(x,wires=range(N_QUBITS),rotation='Z')
 return qml.state()
start=time.time();Str=np.asarray([state(z) for z in Xt]);Ste=np.asarray([state(z) for z in Xe])
Ktr=np.clip(np.abs(Str@Str.conj().T)**2,0,1);Kte=np.clip(np.abs(Ste@Str.conj().T)**2,0,1)
qsvm=SVC(C=1,kernel='precomputed');qsvm.fit(Ktr,y[qtrain]);pred=qsvm.predict(Kte)
record('QSVM',y[qtest],pred,time.time()-start,qtest)
# VQC, trained only on batch 1-7, tuned by validation batch 8
qdev2=qml.device('default.qubit',wires=N_QUBITS)
@qml.qnode(qdev2,interface='torch',diff_method='backprop')
def circuit(inputs,weights):
 qml.AngleEmbedding(inputs,wires=range(N_QUBITS),rotation='Y')
 qml.StronglyEntanglingLayers(weights,wires=range(N_QUBITS))
 return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]
class VQC(nn.Module):
 def __init__(self):
  super().__init__();self.quantum=qml.qnn.TorchLayer(circuit,{'weights':(LAYERS,N_QUBITS,3)});self.head=nn.Linear(N_QUBITS,6)
 def forward(self,x):return self.head(self.quantum(x).float())
model=VQC();opt=torch.optim.Adam(model.parameters(),lr=.02);loss_fn=nn.CrossEntropyLoss()
Xvt=torch.tensor(angles(vtrain));Yvt=torch.tensor(y[vtrain],dtype=torch.long)
Xval=torch.tensor(angles(val_idx));Yval=torch.tensor(y[val_idx],dtype=torch.long)
loader=torch.utils.data.DataLoader(torch.utils.data.TensorDataset(Xvt,Yvt),batch_size=BATCH_SIZE,shuffle=True)
best=-1;best_epoch=0;history=[];start=time.time();checkpoint=OUT/'VQC_best.pt'
for epoch in range(VQC_EPOCHS):
 model.train();total=0
 for xx,yy in loader:
  opt.zero_grad();logits=model(xx);loss=loss_fn(logits,yy);loss.backward();opt.step();total+=loss.item()*len(xx)
 model.eval()
 with torch.no_grad():vp=np.concatenate([model(z).argmax(1).numpy() for z in Xval.split(32)])
 vf=f1_score(y[val_idx],vp,average='macro',zero_division=0);history.append(dict(epoch=epoch+1,train_loss=total/len(vtrain),validation_macro_f1=vf))
 print(f'VQC epoch {epoch+1}: validation macro F1={vf:.4f}')
 if vf>best+1e-5:best=vf;best_epoch=epoch+1;torch.save(model.state_dict(),checkpoint)
 if epoch+1-best_epoch>=4:break
model.load_state_dict(torch.load(checkpoint,map_location='cpu',weights_only=True));model.eval()
with torch.no_grad():pred=np.concatenate([model(z).argmax(1).numpy() for z in torch.tensor(Xe).split(32)])
record('VQC',y[qtest],pred,time.time()-start,qtest)
pd.DataFrame(records).to_csv(OUT/'model_comparison.csv',index=False)
pd.DataFrame(predictions).to_csv(OUT/'test_predictions.csv',index=False)
pd.DataFrame(history).to_csv(OUT/'vqc_learning_curve.csv',index=False)
pd.DataFrame({'sample_index':np.r_[qtrain,val_idx,qtest],'partition':['train']*len(qtrain)+['validation']*len(val_idx)+['test']*len(qtest)}).to_csv(OUT/'split_manifest.csv',index=False)
with open(OUT/'configuration.json','w') as f:json.dump(dict(seed=SEED,qubits=N_QUBITS,layers=LAYERS,qsvm_train=len(qtrain),vqc_train=len(vtrain),test=len(qtest),validation=len(val_idx),train_batches='1-7',validation_batch=8,test_batches='9-10',note='SVM and QSVM matched training subset; VQC has a larger training subset. All use same test samples.'),f,indent=2)
fig,ax=plt.subplots(figsize=(7,4));df=pd.DataFrame(records);ax.bar(df.model,df.macro_f1);ax.set_ylim(0,1);ax.set_ylabel('Macro F1');ax.set_title('Q2: Chronological batch generalization');fig.tight_layout();fig.savefig(OUT/'model_comparison.png',dpi=160);plt.close(fig)
archive=Path('/kaggle/working/Q2_Drift_Robust_QML.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
 for p in OUT.rglob('*'):
  if p.is_file():z.write(p,p.relative_to(OUT))
print('\nRESULTS\n',df.to_string(index=False));print('ZIP:',archive)
try:
 from IPython.display import display,FileLink
 display(FileLink(str(archive)))
except Exception:pass


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 82.7 MB/s eta 0:00:00
PyTorch: 2.11.0+cpu PennyLane: 0.45.1 CUDA available: False (quantum simulator uses CPU)
Loaded: (13910, 128) batches: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)] class counts: [2565 2926 1641 1936 3009 1833]
SVM: accuracy=0.4542, macro_F1=0.4348, time=0.0s
QSVM: accuracy=0.3125, ma

/kaggle/working/Q2_Drift_Robust_QML.zip

In [2]:
# Q2 Stage 2 — Single-cell Kaggle script: CORAL + quantum kernel SVM
# Dataset: UCI Gas Sensor Array Drift; transductive, unlabeled-target adaptation
import os, sys, subprocess, zipfile, io, re, json, time, random, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
for module, package in [('pennylane','pennylane'),('requests','requests')]:
    try: __import__(module)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','-q',package])
import numpy as np, pandas as pd, requests, torch, pennylane as qml
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

# -------- CONFIG --------
SEEDS=[42,123,2026]       # change to [42,123,2026,7,99] for final study
QUBITS=4                 # 4 recommended; 6 increases quantum simulation cost
Q_DEPTH=2
N_TRAIN=400              # QSVM kernel is quadratic in training set size
N_VAL=200
N_TEST=240
RIDGE=1e-3
OUT=Path('/kaggle/working/Q2_Stage2_CORAL_Research')
OUT.mkdir(parents=True,exist_ok=True)
print('PennyLane',qml.__version__,'| Torch CUDA',torch.cuda.is_available(),'| quantum state simulator: CPU')

# -------- LOAD UCI DATA --------
def nested_dat(data, depth=0):
    if depth>5: return []
    out=[]
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        for item in z.infolist():
            if item.is_dir(): continue
            if item.filename.lower().endswith('.dat'): out.append((item.filename,z.read(item)))
            elif item.filename.lower().endswith('.zip'):
                try: out.extend(nested_dat(z.read(item),depth+1))
                except Exception: pass
    return out
files=[]
for p in Path('/kaggle/input').rglob('*'):
    if not p.is_file(): continue
    if p.suffix.lower()=='.dat': files.append((str(p),p.read_bytes()))
    elif p.suffix.lower()=='.zip':
        try: files.extend(nested_dat(p.read_bytes()))
        except Exception: pass
if not files:
    for url in ['https://archive.ics.uci.edu/static/public/224/gas+sensor+array+drift+dataset.zip',
                'https://archive.ics.uci.edu/ml/machine-learning-databases/00224/Dataset.zip']:
        try:
            r=requests.get(url,timeout=90);r.raise_for_status();files=nested_dat(r.content)
            if files: break
        except Exception as e: print('Download unavailable:',str(e)[:120])
if not files: raise RuntimeError('Attach the official UCI Gas Sensor Array Drift ZIP to Kaggle, or enable Internet.')
# Avoid duplicate data if multiple copies were attached
unique={}
for name,blob in files:
    match=re.search(r'batch[_\s-]*(\d+)',name,re.I)
    if match: unique[int(match.group(1))]=(name,blob)
if len(unique)<10: raise RuntimeError(f'Expected 10 batch files, found {sorted(unique)}. Attach complete UCI dataset.')
X=[]; Y=[]; B=[]
for b,(name,blob) in sorted(unique.items()):
    for line in blob.decode('utf-8',errors='replace').splitlines():
        tokens=line.split()
        if not tokens: continue
        try:
            lab=int(float(tokens[0]));vec=np.zeros(128,dtype=np.float64)
            for tok in tokens[1:]:
                if ':' in tok:
                    j,v=tok.split(':',1);j=int(j)-1
                    if 0<=j<128: vec[j]=float(v)
            X.append(vec);Y.append(lab);B.append(b)
        except (ValueError,IndexError): continue
X=np.asarray(X); Y=np.asarray(Y); B=np.asarray(B)
classes=np.unique(Y); mapping={c:i for i,c in enumerate(classes)}; y=np.array([mapping[v] for v in Y])
assert len(classes)==6, f'Expected 6 classes, got {classes}'
print('Dataset:',X.shape,'batch sizes:',dict(zip(*np.unique(B,return_counts=True))))

# -------- SPLITS AND CORAL --------
source=np.where(B<=7)[0]; val=np.where(B==8)[0]; test=np.where(B>=9)[0]
assert len(source)>0 and len(val)>0 and len(test)>0

def sample_ids(ids,n,seed):
    if len(ids)<=n: return ids
    try: picked,_=train_test_split(ids,train_size=n,random_state=seed,stratify=y[ids]);return np.sort(picked)
    except ValueError: return np.sort(np.random.default_rng(seed).choice(ids,n,replace=False))

def matrix_power_sym(A,power,eps=1e-9):
    w,v=np.linalg.eigh((A+A.T)/2)
    return (v*np.maximum(w,eps)**power)@v.T

def coral_source_to_target(src,tgt,ridge=RIDGE):
    # Align labeled source features to the covariance of UNLABELED target features.
    # No target labels are used. The target feature set is transductively available.
    sm=src.mean(axis=0);tm=tgt.mean(axis=0)
    cs=np.cov(src,rowvar=False)+ridge*np.eye(src.shape[1])
    ct=np.cov(tgt,rowvar=False)+ridge*np.eye(src.shape[1])
    return (src-sm)@matrix_power_sym(cs,-0.5)@matrix_power_sym(ct,0.5)+tm

# quantum feature map, statevector overlaps (pure-state fidelity)
qdev=qml.device('default.qubit',wires=QUBITS)
@qml.qnode(qdev)
def q_state(x):
    for layer in range(Q_DEPTH):
        for wire in range(QUBITS):
            qml.RY(x[wire]*(layer+1),wires=wire)
            qml.RZ(0.5*x[wire],wires=wire)
        for wire in range(QUBITS-1): qml.CNOT(wires=[wire,wire+1])
    return qml.state()

def embed(arr):
    return np.asarray([q_state(row) for row in arr],dtype=np.complex128)

def quantum_gram(A,B):
    return np.abs(A@B.conj().T)**2

# All four models use the SAME train, validation, and test subsets.
# Model selection on batch 8; final test batches 9-10 evaluated only afterward.
records=[]; predictions=[]; chosen=[]
for seed in SEEDS:
    print('\n'+'='*65+'\nSeed',seed)
    tr=sample_ids(source,N_TRAIN,seed)
    va=sample_ids(val,N_VAL,seed)
    te=sample_ids(test,N_TEST,seed)
    # Source-only feature fitting prevents target distribution leakage in preprocessing.
    scaler=StandardScaler().fit(X[tr]); xs=scaler.transform(X[tr]);xv=scaler.transform(X[va]);xt=scaler.transform(X[te])
    pca=PCA(n_components=QUBITS,random_state=seed).fit(xs)
    xs=pca.transform(xs);xv=pca.transform(xv);xt=pca.transform(xt)
    # Normalize using SOURCE training statistics only, then clip quantum angles.
    center=xs.mean(0);spread=xs.std(0)+1e-6
    xs=np.clip((xs-center)/spread,-3,3)
    xv=np.clip((xv-center)/spread,-3,3)
    xt=np.clip((xt-center)/spread,-3,3)
    # Unlabeled target feature adaptation, fitted separately for validation and test.
    # Using test FEATURES is transductive; no test LABELS are accessed for CORAL.
    xs_val_c=coral_source_to_target(xs,xv)
    xs_test_c=coral_source_to_target(xs,xt)
    features={
        'SVM':(xs,xv,xs,xt),
        'SVM_CORAL':(xs_val_c,xv,xs_test_c,xt),
        'QSVM':(xs,xv,xs,xt),
        'QSVM_CORAL':(xs_val_c,xv,xs_test_c,xt),
    }
    for name,(trv,valv,trt,tst) in features.items():
        quantum=name.startswith('QSVM')
        best=None
        # Scale features to angles for quantum models. Classical SVM uses same angles
        # so that kernel family, rather than input scale, is the key difference.
        trv=trv*np.pi/3;valv=valv*np.pi/3;trt=trt*np.pi/3;tst=tst*np.pi/3
        t0=time.time()
        if quantum:
            # Precompute quantum statevectors once per split/configuration.
            sv=embed(trv);vv=embed(valv)
            Ktr=quantum_gram(sv,sv);Kval=quantum_gram(vv,sv)
        for C in [0.1,1,10]:
            if quantum:
                model=SVC(C=C,kernel='precomputed',class_weight='balanced')
                model.fit(Ktr,y[tr]); pv=model.predict(Kval)
            else:
                model=SVC(C=C,kernel='rbf',gamma='scale',class_weight='balanced')
                model.fit(trv,y[tr]);pv=model.predict(valv)
            score=f1_score(y[va],pv,average='macro',zero_division=0)
            if best is None or score>best[0]: best=(score,C)
        val_f1,C=best
        if quantum:
            st=embed(trt);tt=embed(tst)
            Ktrain=quantum_gram(st,st);Ktest=quantum_gram(tt,st)
            clf=SVC(C=C,kernel='precomputed',class_weight='balanced').fit(Ktrain,y[tr]);pred=clf.predict(Ktest)
        else:
            clf=SVC(C=C,kernel='rbf',gamma='scale',class_weight='balanced').fit(trt,y[tr]);pred=clf.predict(tst)
        duration=time.time()-t0
        result={'seed':seed,'model':name,'C':C,'validation_macro_f1':val_f1,
                'accuracy':accuracy_score(y[te],pred),'macro_f1':f1_score(y[te],pred,average='macro',zero_division=0),
                'macro_precision':precision_score(y[te],pred,average='macro',zero_division=0),
                'macro_recall':recall_score(y[te],pred,average='macro',zero_division=0),
                'seconds':duration,'train_n':len(tr),'val_n':len(va),'test_n':len(te)}
        records.append(result)
        print(name,'C',C,'val F1',round(val_f1,4),'test acc',round(result['accuracy'],4),'test F1',round(result['macro_f1'],4),'sec',round(duration,1))
        for idx,p in zip(te,pred): predictions.append({'seed':seed,'model':name,'row_index':int(idx),'batch':int(B[idx]),'true':int(y[idx]),'pred':int(p)})
        if seed==SEEDS[0]:
            cm=confusion_matrix(y[te],pred,labels=np.arange(6))
            fig,ax=plt.subplots(figsize=(7,6))
            ConfusionMatrixDisplay(cm,display_labels=[str(c) for c in classes]).plot(ax=ax,cmap='Blues',colorbar=False)
            ax.set_title(f'{name} | Chronological drift | seed {seed}');plt.tight_layout()
            fig.savefig(OUT/f'confusion_{name}.png',dpi=180);plt.close(fig)
            (OUT/f'report_{name}.txt').write_text(classification_report(y[te],pred,labels=np.arange(6),zero_division=0))
    pd.DataFrame(records).to_csv(OUT/'all_seed_results.csv',index=False)
    pd.DataFrame(predictions).to_csv(OUT/'predictions.csv',index=False)

res=pd.DataFrame(records)
summary=res.groupby('model').agg(accuracy_mean=('accuracy','mean'),accuracy_sd=('accuracy','std'),macro_f1_mean=('macro_f1','mean'),macro_f1_sd=('macro_f1','std'),runtime_mean=('seconds','mean')).reset_index()
summary.to_csv(OUT/'summary.csv',index=False)
# Paired per-seed improvements, NOT independent-sample significance tests.
wide=res.pivot(index='seed',columns='model',values='macro_f1')
paired=pd.DataFrame({'seed':wide.index,'CORAL_minus_SVM':wide['SVM_CORAL']-wide['SVM'],
                     'CORAL_minus_QSVM':wide['QSVM_CORAL']-wide['QSVM'],
                     'QSVM_CORAL_minus_SVM_CORAL':wide['QSVM_CORAL']-wide['SVM_CORAL']})
paired.to_csv(OUT/'paired_differences.csv',index=False)
fig,ax=plt.subplots(figsize=(9,5));ax.bar(summary['model'],summary['macro_f1_mean'],yerr=summary['macro_f1_sd'].fillna(0),capsize=4)
ax.set_ylim(0,1);ax.set_ylabel('Mean macro F1 (± SD across seeds)');ax.set_title('Q2 Stage 2: Chronological drift comparison');plt.tight_layout();fig.savefig(OUT/'comparison.png',dpi=180);plt.close(fig)
metadata={'dataset':'UCI Gas Sensor Array Drift','splits':{'source':'batches 1-7','validation':'batch 8','test':'batches 9-10'},
          'adaptation':'transductive CORAL, target features only, no target labels',
          'seeds':SEEDS,'qubits':QUBITS,'depth':Q_DEPTH,'train_subset':N_TRAIN,'validation_subset':N_VAL,'test_subset':N_TEST,
          'kernel':'squared quantum state overlap; noiseless statevector simulator',
          'limitation':'This evaluates kernel-based CORAL adaptation, not hardware advantage; test features used unlabeled (transductive).'}
(OUT/'experiment_config.json').write_text(json.dumps(metadata,indent=2))
zip_path=Path('/kaggle/working/Q2_Stage2_CORAL_Research.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file(): z.write(p,arcname=p.relative_to(OUT))
print('\nFINAL SUMMARY\n',summary.to_string(index=False))
print('\nSaved:',zip_path)
try:
    from IPython.display import display,FileLink
    display(FileLink(str(zip_path)))
except Exception: pass


PennyLane 0.45.1 | Torch CUDA False | quantum state simulator: CPU
Dataset: (13910, 128) batch sizes: {np.int64(1): np.int64(445), np.int64(2): np.int64(1244), np.int64(3): np.int64(1586), np.int64(4): np.int64(161), np.int64(5): np.int64(197), np.int64(6): np.int64(2300), np.int64(7): np.int64(3613), np.int64(8): np.int64(294), np.int64(9): np.int64(470), np.int64(10): np.int64(3600)}

Seed 42
SVM C 10 val F1 0.4248 test acc 0.4125 test F1 0.3702 sec 0.1
SVM_CORAL C 10 val F1 0.5577 test acc 0.4875 test F1 0.4629 sec 0.1
QSVM C 10 val F1 0.4627 test acc 0.3792 test F1 0.3838 sec 4.7
QSVM_CORAL C 10 val F1 0.6117 test acc 0.4583 test F1 0.4429 sec 4.5

Seed 123
SVM C 10 val F1 0.3251 test acc 0.4083 test F1 0.403 sec 0.1
SVM_CORAL C 1 val F1 0.5792 test acc 0.425 test F1 0.3827 sec 0.1
QSVM C 1 val F1 0.3569 test acc 0.2833 test F1 0.2663 sec 4.2
QSVM_CORAL C 1 val F1 0.5092 test acc 0.325 test F1 0.3027 sec 4.4

Seed 2026
SVM C 10 val F1 0.6355 test acc 0.5708 test F1 0.5702 sec 0.1
S

/kaggle/working/Q2_Stage2_CORAL_Research.zip

In [1]:
# Q2 STAGE 3 — ONE CELL KAGGLE RESEARCH EXPERIMENT
# Chronological gas sensor drift: SVM vs quantum-kernel SVM; None / CORAL / MMD
# Unlabeled target features used for transductive adaptation. NEVER target labels.
import os, io, re, json, time, zipfile, hashlib, warnings, random
from pathlib import Path
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.linalg import eigh
from scipy.stats import t as student_t, wilcoxon
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.metrics.pairwise import rbf_kernel

# ==================== USER CONFIGURATION ====================
SEEDS = [42, 123, 2026, 7, 99]
QUBITS = 4             # Phase A: 4; run 6 or 8 separately for ablations
Q_DEPTH = 2            # Phase A: 2
N_TRAIN = 400          # Matched training sample count for both classifiers
N_VAL = 200            # Batch 8: validation labels only, never final-test labels
N_TEST = None          # None = ALL samples in batches 9 and 10
C_GRID = [0.1, 1.0, 10.0]
MMD_STEPS = 120        # Affine feature alignment, unsupervised
MMD_LR = 0.025
MMD_REG = 0.12
MMD_FIT_TARGET_MAX = 900   # Unlabeled target subsample for adaptation only
RIDGE = 1e-3
OUTPUT = Path('/kaggle/working/Q2_Stage3_Research') if Path('/kaggle').exists() else Path('Q2_Stage3_Research')
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Q2 Stage 3 | output:', OUTPUT.resolve())

# ==================== DATA LOADER ====================
def nested_dat(blob, depth=0):
    if depth > 5: return []
    found=[]
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        for item in z.infolist():
            if item.is_dir(): continue
            if item.filename.lower().endswith('.dat'):
                found.append((item.filename,z.read(item)))
            elif item.filename.lower().endswith('.zip'):
                try: found.extend(nested_dat(z.read(item),depth+1))
                except (zipfile.BadZipFile, ValueError): pass
    return found

files=[]
for root in ['/kaggle/input', '/mnt/data']:
    p0=Path(root)
    if not p0.exists(): continue
    for p in p0.rglob('*'):
        if not p.is_file(): continue
        if p.suffix.lower()=='.dat': files.append((str(p),p.read_bytes()))
        elif p.suffix.lower()=='.zip':
            try: files.extend(nested_dat(p.read_bytes()))
            except (zipfile.BadZipFile, ValueError): pass
if not files:
    import requests
    for url in ['https://archive.ics.uci.edu/static/public/224/gas+sensor+array+drift+dataset.zip',
                'https://archive.ics.uci.edu/ml/machine-learning-databases/00224/Dataset.zip']:
        try:
            r=requests.get(url,timeout=90);r.raise_for_status()
            files=nested_dat(r.content)
            if files: break
        except Exception as e: print('Download attempt:',str(e)[:100])
unique={}
for name,blob in files:
    m=re.search(r'batch[_\s-]*(\d+)',name,re.I)
    if m: unique[int(m.group(1))]=(name,blob)
if len(unique)!=10: raise RuntimeError('Need all 10 UCI batch*.dat files. Attach official UCI Dataset.zip to Kaggle or enable Internet. Found: '+str(sorted(unique)))
X=[];Y=[];B=[]
for b,(name,blob) in sorted(unique.items()):
    for line in blob.decode('utf-8',errors='replace').splitlines():
        fields=line.split()
        if not fields: continue
        try:
            label=int(float(fields[0]));row=np.zeros(128,dtype=np.float64)
            for token in fields[1:]:
                if ':' not in token: continue
                i,v=token.split(':',1);i=int(i)-1
                if 0<=i<128: row[i]=float(v)
            X.append(row);Y.append(label);B.append(b)
        except (ValueError,IndexError): pass
X=np.asarray(X);Y=np.asarray(Y);B=np.asarray(B)
classes=np.sort(np.unique(Y)); assert len(classes)==6
labels=np.searchsorted(classes,Y)
source_ids=np.where(B<=7)[0];val_ids=np.where(B==8)[0];test_ids=np.where(B>=9)[0]
print('Dataset:',X.shape,'| source',len(source_ids),'validation',len(val_ids),'test',len(test_ids))

# ==================== ALIGNMENT ====================
def sample_ids(ids,n,seed):
    if n is None or len(ids)<=n: return np.sort(ids)
    try: return np.sort(train_test_split(ids,train_size=n,random_state=seed,stratify=labels[ids])[0])
    except ValueError: return np.sort(np.random.default_rng(seed).choice(ids,n,replace=False))

def sym_power(A,p):
    w,v=eigh((A+A.T)/2)
    return (v*np.maximum(w,1e-9)**p)@v.T

def coral(src,tgt,ridge=RIDGE):
    ms=src.mean(0);mt=tgt.mean(0)
    cs=np.cov(src,rowvar=False)+ridge*np.eye(src.shape[1])
    ct=np.cov(tgt,rowvar=False)+ridge*np.eye(src.shape[1])
    return (src-ms)@sym_power(cs,-0.5)@sym_power(ct,0.5)+mt

def mmd_align(src,tgt,seed):
    # Minimize multi-bandwidth RBF-MMD by learning a regularized affine
    # source transform; target labels are not used. Requires torch.
    import torch
    torch.manual_seed(seed)
    rng=np.random.default_rng(seed)
    if len(tgt)>MMD_FIT_TARGET_MAX:
        tgt=tgt[rng.choice(len(tgt),MMD_FIT_TARGET_MAX,replace=False)]
    if len(src)>MMD_FIT_TARGET_MAX:
        src_fit=src[rng.choice(len(src),MMD_FIT_TARGET_MAX,replace=False)]
    else: src_fit=src
    # Scale distances to stabilize MMD across domains
    scale=max(float(np.median(np.linalg.norm(src_fit[:min(150,len(src_fit))]-src_fit[0],axis=1))),1e-2)
    a=torch.tensor(src_fit/scale,dtype=torch.float32)
    b=torch.tensor(tgt/scale,dtype=torch.float32)
    d=src.shape[1]
    W=torch.nn.Parameter(torch.eye(d)); bias=torch.nn.Parameter(torch.zeros(d))
    opt=torch.optim.Adam([W,bias],lr=MMD_LR)
    def kern(x,z):
        dist=torch.cdist(x,z).square()
        return sum(torch.exp(-dist/(2*s*s)) for s in (0.5,1.,2.,4.))/4
    # Fixed deterministic 160-point subsets reduce MMD optimization cost
    ai=torch.randperm(len(a))[:min(160,len(a))]
    bi=torch.randperm(len(b))[:min(160,len(b))]
    aa=a[ai];bb=b[bi]
    for _ in range(MMD_STEPS):
        opt.zero_grad(); aa2=aa@W+bias
        loss=kern(aa2,aa2).mean()+kern(bb,bb).mean()-2*kern(aa2,bb).mean()
        loss=loss+MMD_REG*((W-torch.eye(d)).square().mean()+bias.square().mean())
        loss.backward();opt.step()
    return ((src/scale)@W.detach().numpy()+bias.detach().numpy())*scale

def adapt(src,tgt,method,seed):
    if method=='None': return src.copy()
    if method=='CORAL': return coral(src,tgt)
    if method=='MMD': return mmd_align(src,tgt,seed)
    raise ValueError(method)

# ==================== QUANTUM FEATURE MAP ====================
# PennyLane pure-state simulation. Squared inner products define a PSD quantum kernel.
try: import pennylane as qml
except ImportError:
    import sys,subprocess
    subprocess.check_call([sys.executable,'-m','pip','install','-q','pennylane'])
    import pennylane as qml
qdev=qml.device('default.qubit',wires=QUBITS)
@qml.qnode(qdev)
def quantum_state(x):
    for depth in range(Q_DEPTH):
        for w in range(QUBITS):
            qml.RY(x[w]*(depth+1),wires=w)
            qml.RZ(x[w]*0.5,wires=w)
        for w in range(QUBITS-1): qml.CNOT(wires=[w,w+1])
    return qml.state()

def q_states(arr):
    # bounded angles; fixed map, no training or test-label information
    angles=2*np.arctan(arr)
    return np.asarray([quantum_state(v) for v in angles],dtype=np.complex128)

def qkernel(A,B):
    return np.abs(A@B.conj().T)**2

# ==================== EXPERIMENT ====================
rows=[];predictions=[];timings=[]
methods=['None','CORAL','MMD']
models=['SVM','QSVM']
for seed in SEEDS:
    print('\nSEED',seed,flush=True)
    rng=np.random.default_rng(seed)
    tr=sample_ids(source_ids,N_TRAIN,seed)
    va=sample_ids(val_ids,N_VAL,seed)
    te=sample_ids(test_ids,N_TEST,seed)
    scaler=StandardScaler().fit(X[tr])
    tr0=scaler.transform(X[tr]);va0=scaler.transform(X[va]);te0=scaler.transform(X[te])
    pca=PCA(n_components=QUBITS,random_state=seed).fit(tr0)
    tr0=pca.transform(tr0);va0=pca.transform(va0);te0=pca.transform(te0)
    # Select C using only batch-8 labels, with adaptation to unlabeled batch-8.
    bestC={}
    for method in methods:
        trv=adapt(tr0,va0,method,seed)
        state_train=q_states(trv);state_val=q_states(va0)
        Ktr=qkernel(state_train,state_train);Kva=qkernel(state_val,state_train)
        for model in models:
            score_best=-np.inf;chosen=C_GRID[0]
            for C in C_GRID:
                if model=='SVM':
                    clf=SVC(C=C,kernel='rbf',gamma='scale',class_weight='balanced').fit(trv,labels[tr])
                    pv=clf.predict(va0)
                else:
                    clf=SVC(C=C,kernel='precomputed',class_weight='balanced').fit(Ktr,labels[tr])
                    pv=clf.predict(Kva)
                score=f1_score(labels[va],pv,average='macro',zero_division=0)
                if score>score_best: score_best=score;chosen=C
            bestC[(method,model)]=chosen
            print(' ',method,model,'validation F1',round(score_best,4),'C',chosen,flush=True)
    # Final evaluation uses ALL target samples; alignment uses their unlabeled features.
    for method in methods:
        t0=time.time()
        trt=adapt(tr0,te0,method,seed)
        qs=q_states(trt);qt=q_states(te0)
        Ktrain=qkernel(qs,qs);Ktest=qkernel(qt,qs)
        for model in models:
            C=bestC[(method,model)]
            if model=='SVM':
                clf=SVC(C=C,kernel='rbf',gamma='scale',class_weight='balanced').fit(trt,labels[tr])
                predicted=clf.predict(te0)
            else:
                clf=SVC(C=C,kernel='precomputed',class_weight='balanced').fit(Ktrain,labels[tr])
                predicted=clf.predict(Ktest)
            record={'seed':seed,'alignment':method,'model':model,'C':C,
                    'n_train':len(tr),'n_val':len(va),'n_test':len(te),
                    'accuracy':accuracy_score(labels[te],predicted),
                    'macro_f1':f1_score(labels[te],predicted,average='macro',zero_division=0),
                    'runtime_shared_alignment_and_kernel_sec':time.time()-t0}
            rows.append(record)
            for idx,actual,pr in zip(te,labels[te],predicted):
                predictions.append({'seed':seed,'alignment':method,'model':model,'original_index':int(idx),
                                    'batch':int(B[idx]),'true':int(actual),'predicted':int(pr)})
            print('  TEST',method,model,'Acc',round(record['accuracy'],4),'F1',round(record['macro_f1'],4),flush=True)
            if seed==SEEDS[0]:
                cm=confusion_matrix(labels[te],predicted,labels=np.arange(6))
                fig,ax=plt.subplots(figsize=(7,6))
                ConfusionMatrixDisplay(cm,display_labels=[str(c) for c in classes]).plot(ax=ax,cmap='Blues',colorbar=False)
                ax.set_title(f'{model} + {method} | seed {seed} | target batches 9-10')
                fig.tight_layout();fig.savefig(OUTPUT/f'confusion_{model}_{method}.png',dpi=170);plt.close(fig)
        # persist after each alignment to reduce lost progress
        pd.DataFrame(rows).to_csv(OUTPUT/'all_seed_results.csv',index=False)
        pd.DataFrame(predictions).to_csv(OUTPUT/'predictions.csv',index=False)

results=pd.DataFrame(rows)
summary=results.groupby(['alignment','model'],as_index=False).agg(
    n_seeds=('macro_f1','count'),accuracy_mean=('accuracy','mean'),accuracy_sd=('accuracy','std'),
    macro_f1_mean=('macro_f1','mean'),macro_f1_sd=('macro_f1','std'))
summary.to_csv(OUTPUT/'summary.csv',index=False)
# Paired differences by seed; CI is descriptive, not evidence of significance by itself.
comparisons=[]
for model in models:
    p=results[results.model==model].pivot(index='seed',columns='alignment',values='macro_f1')
    for method in ['CORAL','MMD']:
        diff=(p[method]-p['None']).dropna().values
        n=len(diff);mean=float(np.mean(diff));sd=float(np.std(diff,ddof=1)) if n>1 else float('nan')
        half=float(student_t.ppf(.975,n-1)*sd/np.sqrt(n)) if n>1 else float('nan')
        comparisons.append({'model':model,'contrast':f'{method} - None','n':n,'mean_delta_f1':mean,
                            'ci95_low':mean-half,'ci95_high':mean+half,
                            'wilcoxon_p_exploratory':float(wilcoxon(diff).pvalue) if n>=5 and np.any(diff) else np.nan})
pd.DataFrame(comparisons).to_csv(OUTPUT/'paired_differences.csv',index=False)
fig,ax=plt.subplots(figsize=(9,5))
summary['label']=summary['model']+' + '+summary['alignment']
ax.bar(summary['label'],summary['macro_f1_mean'],yerr=summary['macro_f1_sd'].fillna(0),capsize=4)
ax.set_ylim(0,1);ax.set_ylabel('Macro F1 (mean ± seed SD)');ax.tick_params(axis='x',rotation=35)
fig.tight_layout();fig.savefig(OUTPUT/'model_comparison.png',dpi=180);plt.close(fig)
config={'seeds':SEEDS,'qubits':QUBITS,'depth':Q_DEPTH,'train_samples':N_TRAIN,'validation_samples':N_VAL,
        'test_samples':N_TEST,'C_grid':C_GRID,'methods':methods,'train_batches':'1-7','validation_batch':8,
        'test_batches':'9-10','adaptation':'transductive: unlabeled target features available before inference',
        'note':'Five seeds represent sampling variability on a fixed batch partition; not five independent datasets. No target test labels used for tuning.',
        'mmd':'regularized affine alignment optimized by Gaussian-kernel MMD; not MMD classifier loss',
        'quantum':'PennyLane CPU statevector squared-fidelity kernel; no hardware speedup claims'}
(OUTPUT/'experiment_config.json').write_text(json.dumps(config,indent=2))
zip_path=OUTPUT.parent/'Q2_Stage3_Research.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for f in OUTPUT.rglob('*'):
        if f.is_file():z.write(f,f.relative_to(OUTPUT))
print('\nFINAL SUMMARY\n',summary.to_string(index=False))
print('\nSaved:',zip_path)
try:
    from IPython.display import display,FileLink
    display(FileLink(str(zip_path)))
except ImportError: pass


Q2 Stage 3 | output: /kaggle/working/Q2_Stage3_Research
Dataset: (13910, 128) | source 9546 validation 294 test 4070
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 121.2 MB/s eta 0:00:00

SEED 42
  None SVM validation F1 0.4065 C 10.0
  None QSVM validation F1 0.3437 C 10.0
  CORAL SVM validation F1 0.483 C 10.0
  CORAL QSVM validation F1 0.2268 C 1.0
  MMD SVM validation F1 0.616 C 10.0
  MMD QSVM validation F1 0.325 C 10.0
  TEST None

/kaggle/working/Q2_Stage3_Research.zip